### **Notebook 4 (etapa 4): Entrenamiento y evaluación de regresión Bootstrapping**

In [ ]:
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y estadísticas (medias, desviaciones estándar)
import os  # Interacción con el sistema operativo (creación de carpetas y manejo de rutas)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM en los bucles
import time  # Medición de los tiempos de ejecución por iteración y del proceso total
from sklearn.ensemble import RandomForestClassifier  # Algoritmo de ensamble utilizado para medir la importancia de variables
from sklearn.metrics import f1_score, average_precision_score  # Métricas para evaluar el rendimiento en las muestras no vistas (OOB)

def seleccion_caracteristicas_bootstrapping_rf_mortalidad():
    """
    Descripción:
        Aplica el método de Stability Selection mediante Bootstrapping (100 repeticiones)
        sobre un modelo regularizado de Random Forest para la variable objetivo 'MORTALIDAD'.
        Permite identificar qué variables son consistentemente importantes aislando el ruido,
        evaluándolas según su frecuencia de aparición y su peso promedio a lo largo de las iteraciones.
        Utiliza los registros no seleccionados (Out-Of-Bag) para evaluar el rendimiento interno del modelo.

    Entradas:
        - Ninguna: La función está autocontenida y lee los datos desde una ruta fija en el código.

    Salidas:
        - None: La función guarda los resultados directamente en disco local:
            1. CSV con el reporte de estabilidad de características (tasa de aparición e importancia).
            2. CSV con el registro de métricas de validación OOB (Out-Of-Bag) de cada iteración.
    """
    # Establecer la variable objetivo y la cantidad de muestras de bootstrap a generar
    target_name = 'MORTALIDAD'
    n_repeticiones = 100
    
    # 1. Configuración inicial y rutas
    dir_datos = "../../Datos/Datasets Finales"
    # Ajustado para guardar ordenadamente en la nueva subcarpeta específica de Bootstrapping
    dir_resultados = "../../Resultados/Resultados (etapa 3 y 4)/Bootstrapping"
    os.makedirs(dir_resultados, exist_ok=True)

    # Lista de variables que no deben ser incluidas como predictores
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60)
    print(f"INICIANDO BOOTSTRAPPING (100 ITERACIONES) PARA: {target_name} (RANDOM FOREST)")
    print("="*60)

    # 2. Cargar datos de entrenamiento
    print("Cargando datasets...")
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control.csv"), low_memory=False)

    # Balancear las clases emparejando el tamaño de la cohorte de control con la oncológica
    print("Balanceando la cohorte basal...")
    n_onco = len(df_onco_train)
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42)
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True)

    # Liberar memoria de los dataframes temporales
    del df_onco_train, df_control_train, df_control_sample
    gc.collect()

    # Definir la lista final de características predictoras
    features = [col for col in df_train_maestro.columns if col not in cols_excluir]
    
    # Inicializar diccionarios y listas para almacenar el historial de las 100 iteraciones
    memoria_importancias = {feature: [] for feature in features}
    frecuencia_seleccion = {feature: 0 for feature in features}
    registro_metricas_oob = []

    print(f"Iniciando bucle de {n_repeticiones} iteraciones")
    # Registrar tiempo de inicio general del bucle
    tiempo_inicio_total = time.time()

    # Guardar los índices originales del dataframe para luego cruzar y calcular los Out-Of-Bag (OOB)
    indices_totales = df_train_maestro.index

    # --- BUCLE PRINCIPAL DE BOOTSTRAPPING ---
    for i in range(n_repeticiones):
        # Medir tiempo por cada iteración individual
        inicio_iter = time.time()
        
        # Generar muestra Bootstrap: 100% de tamaño original, permitiendo registros repetidos (con reemplazo)
        df_boot = df_train_maestro.sample(frac=1.0, replace=True, random_state=i)
        
        # Identificar registros que quedaron fuera de la muestra para usarlos como set de validación (OOB)
        indices_oob = indices_totales.difference(df_boot.index)
        df_oob = df_train_maestro.loc[indices_oob]
        
        # Separar variables independientes (X) y dependiente (y) para la muestra Bootstrap
        X_boot = df_boot[features]
        y_boot = df_boot[target_name]
        
        # Separar variables independientes (X) y dependiente (y) para la muestra OOB (validación)
        X_oob = df_oob[features]
        y_oob = df_oob[target_name]
        
        # MODELO ACTUALIZADO CON LOS HIPERPARÁMETROS GANADORES EN MODELADO (Estables)
        modelo_rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=35,           # <--- CONFIGURACIÓN ESTABLE ENCONTRADA (PODA AUTOMÁTICA)
            min_samples_split=10,
            class_weight='balanced',
            n_jobs=-1,              # Utiliza todos los vCPUs asignados al servidor para agilizar
            random_state=42
        )
        # Entrenar el modelo estrictamente con la muestra de Bootstrap actual
        modelo_rf.fit(X_boot, y_boot)
        
        # Extraer el nivel de importancia que el modelo le dio a cada variable en esta corrida
        importancias = modelo_rf.feature_importances_
        
        # Guardar en memoria el aporte de cada variable
        for idx, col in enumerate(features):
            imp_val = importancias[idx]
            memoria_importancias[col].append(imp_val) # Guarda el valor numérico
            if imp_val > 0:
                frecuencia_seleccion[col] += 1 # Suma +1 si la variable aportó información (no fue ignorada)
                
        # Validar el modelo usando los datos OOB que el árbol no ha visto en esta iteración
        y_pred_oob = modelo_rf.predict(X_oob)
        y_prob_oob = modelo_rf.predict_proba(X_oob)[:, 1]
        
        # Calcular métricas del desempeño sobre la porción OOB
        f1_macro_oob = f1_score(y_oob, y_pred_oob, average='macro')
        auprc_oob = average_precision_score(y_oob, y_prob_oob)
        
        # Almacenar métricas de esta iteración para el reporte final
        registro_metricas_oob.append({'Iteracion': i+1, 'F1_Macro_OOB': f1_macro_oob, 'AUPRC_OOB': auprc_oob})
        
        # Marcar fin del ciclo y notificar progreso en consola
        fin_iter = time.time()
        print(f"   -> Iteración {i+1}/{n_repeticiones} completada en {round(fin_iter - inicio_iter, 1)} seg. | OOB F1-Macro: {f1_macro_oob:.4f}")

        # ==========================================================
        # AGREGADO DE SEGURIDAD (Destrucción de variables pesadas)
        # ==========================================================
        # Eliminar las matrices creadas en el ciclo actual para evitar desbordamientos de memoria (Memory Leaks)
        del df_boot, df_oob, X_boot, y_boot, X_oob, y_oob
        # Forzar la limpieza de RAM profunda cada 10 iteraciones
        if (i + 1) % 10 == 0:
            gc.collect() 

    # Evaluar tiempo total consumido por el proceso de Bootstrapping
    tiempo_fin_total = time.time()
    print(f"\nÉXITO: Bootstrapping completado en {round((tiempo_fin_total - tiempo_inicio_total)/60, 2)} minutos.")

    # FASE 3: CONSOLIDACIÓN Y ALMACENAMIENTO DE RESULTADOS
    resultados_finales = []
    # Calcular promedios, varianzas y porcentajes consolidados para cada variable a lo largo del proceso
    for col in features:
        tasa_aparicion = (frecuencia_seleccion[col] / n_repeticiones) * 100
        imp_promedio = np.mean(memoria_importancias[col])
        imp_std = np.std(memoria_importancias[col])
        
        resultados_finales.append({
            'Variable': col,
            'Tasa_Selection_Porcentaje': tasa_aparicion,
            'Importancia_Promedio': imp_promedio,
            'Varianza_Importancia': imp_std
        })
        
    # Transformar la lista de resultados a DataFrame y ordenarlo por tasa de aparición (mayor a menor)
    df_resultados = pd.DataFrame(resultados_finales)
    df_resultados = df_resultados.sort_values(by=['Tasa_Selection_Porcentaje', 'Importancia_Promedio'], ascending=[False, False])
    
    # Exportar el ranking de estabilidad de variables a CSV
    ruta_vars = os.path.join(dir_resultados, f"Estabilidad_Variables_RF_{target_name}.csv")
    df_resultados.to_csv(ruta_vars, index=False)
    
    # Exportar el historial de rendimiento en Out-Of-Bag de cada iteración a CSV
    df_metricas = pd.DataFrame(registro_metricas_oob)
    ruta_metricas = os.path.join(dir_resultados, f"Metricas_OOB_RF_{target_name}.csv")
    df_metricas.to_csv(ruta_metricas, index=False)
    
    # Imprimir resumen de guardado y métrica promedio final por consola
    print("="*60)
    print(f"- Reporte de estabilidad guardado en: {ruta_vars}")
    print(f"- Métricas OOB guardadas en: {ruta_metricas}")
    print(f"Métrica OOB promedio final -> F1-Macro: {df_metricas['F1_Macro_OOB'].mean():.4f} | AUPRC: {df_metricas['AUPRC_OOB'].mean():.4f}")
    print("="*60)

In [ ]:
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y estadísticas (medias, desviaciones estándar)
import os  # Interacción con el sistema operativo (creación de carpetas y manejo de rutas)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM en los bucles
import time  # Medición de los tiempos de ejecución por iteración y del proceso total
import xgboost as xgb  # Algoritmo de ensamble avanzado (Gradient Boosting) para medir la importancia de variables
from sklearn.metrics import f1_score, average_precision_score  # Métricas para evaluar el rendimiento en muestras no vistas (OOB)
from sklearn.preprocessing import label_binarize  # Convierte etiquetas multiclase en un formato binario (One-vs-Rest) para métricas de probabilidad

def seleccion_caracteristicas_bootstrapping_xgb_multiclase(target_name):
    """
    Descripción:
        Aplica el método de Stability Selection mediante Bootstrapping (100 repeticiones)
        sobre un modelo optimizado de XGBoost específicamente para variables MULTICLASE.
        Evalúa la consistencia de las variables predictoras aislando el ruido, calculando 
        su tasa de selección y su peso promedio a través de múltiples submuestras aleatorias.
        Estructura las salidas y métricas Out-Of-Bag (OOB) en la subcarpeta consolidada.

    Entradas:
        - target_name (str): Nombre de la variable objetivo multiclase a predecir (ej. 'SEVERIDAD' o 'CONSUMO_RECURSOS').

    Salidas:
        - None: La función guarda los resultados directamente en disco local:
            1. CSV con el reporte de estabilidad de características (tasa de aparición e importancia).
            2. CSV con el registro de métricas de validación OOB (Out-Of-Bag) de cada iteración.
    """
    # Establecer la cantidad de muestras de bootstrap a generar
    n_repeticiones = 100
    
    # 1. Configuración inicial y rutas
    dir_datos = "../../Datos/Datasets Finales"
    
    # Ruta homologada y ordenada para la subcarpeta independiente de Bootstrapping
    dir_resultados = "../../Resultados/Resultados (etapa 3 y 4)/Bootstrapping"
    os.makedirs(dir_resultados, exist_ok=True)

    # Lista de variables que no deben ser incluidas como predictores en el modelo
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']

    print("="*60)
    print(f"INICIANDO BOOTSTRAPPING MULTICLASE PARA: {target_name} (XGBOOST)")
    print("="*60)

    # 2. Cargar datos de entrenamiento
    print("Cargando datasets...")
    # Lectura del dataset con casos oncológicos
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
    # Lectura del dataset con casos de control
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control.csv"), low_memory=False)

    # Balancear las clases emparejando el tamaño de la cohorte de control con la oncológica
    print("Balanceando la cohorte basal...")
    n_onco = len(df_onco_train)
    # Tomar una muestra aleatoria de controles equivalente al tamaño oncológico
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42)
    # Unificar los subconjuntos para crear el dataframe maestro
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True)

    # Liberar memoria de los dataframes temporales
    del df_onco_train, df_control_train, df_control_sample
    gc.collect()

    # Definir la lista final de características predictoras (filtrando las excluidas)
    features = [col for col in df_train_maestro.columns if col not in cols_excluir]
    # Identificar todas las clases únicas presentes en la variable objetivo multiclase
    clases_unicas = np.unique(df_train_maestro[target_name])

    # Inicializar diccionarios y listas para almacenar el historial de las 100 iteraciones
    memoria_importancias = {feature: [] for feature in features}
    frecuencia_seleccion = {feature: 0 for feature in features}
    registro_metricas_oob = []

    print(f"Iniciando bucle de {n_repeticiones} iteraciones...")
    # Registrar tiempo de inicio general del bucle
    tiempo_inicio_total = time.time()
    # Guardar los índices originales del dataframe para luego identificar los Out-Of-Bag (OOB)
    indices_totales = df_train_maestro.index

    # --- BUCLE PRINCIPAL DE BOOTSTRAPPING ---
    for i in range(n_repeticiones):
        # Medir tiempo por cada iteración individual
        inicio_iter = time.time()
        
        # Generar muestra Bootstrap: 100% de tamaño original, permitiendo registros repetidos (con reemplazo)
        df_boot = df_train_maestro.sample(frac=1.0, replace=True, random_state=i)
        
        # Identificar registros que quedaron fuera de la muestra para usarlos como set de validación (OOB)
        indices_oob = indices_totales.difference(df_boot.index)
        df_oob = df_train_maestro.loc[indices_oob]
        
        # Separar variables independientes (X) y dependiente (y) para la muestra Bootstrap
        X_boot = df_boot[features]
        y_boot = df_boot[target_name]
        
        # Separar variables independientes (X) y dependiente (y) para la muestra OOB (validación)
        X_oob = df_oob[features]
        y_oob = df_oob[target_name]
        
        # MODELO CONFIGURADO CON LOS HIPERPARÁMETROS ÓPTIMOS ENCONTRADOS
        modelo_xgb = xgb.XGBClassifier(
            learning_rate=0.3,      # Configuración estable ganadora de la etapa de modelado
            max_depth=10,           # Configuración estable ganadora de la etapa de modelado
            tree_method='hist',     # Algoritmo basado en histogramas para mayor velocidad
            n_jobs=-1,              # Explota la capacidad multihilo del Droplet (servidor)
            random_state=42         # Semilla para reproducibilidad interna
        )
        # Entrenar el modelo estrictamente con la muestra de Bootstrap actual
        modelo_xgb.fit(X_boot, y_boot)
        
        # Extraer el nivel de importancia (pesos internos) que el modelo asignó a cada variable
        importancias = modelo_xgb.feature_importances_
        
        # Guardar en memoria el aporte de cada variable
        for idx, col in enumerate(features):
            imp_val = importancias[idx]
            memoria_importancias[col].append(imp_val) # Guarda el valor numérico
            if imp_val > 0:
                frecuencia_seleccion[col] += 1 # Suma +1 si la variable aportó información (no fue ignorada)
                
        # Validar el modelo usando los datos OOB que los árboles no han visto en esta iteración
        y_pred_oob = modelo_xgb.predict(X_oob)
        y_prob_oob = modelo_xgb.predict_proba(X_oob)
        
        # Calcular la métrica F1-Macro sobre la porción OOB
        f1_macro_oob = f1_score(y_oob, y_pred_oob, average='macro')
        
        # Binarizar las clases reales para calcular el AUPRC ponderado (estrategia One-vs-Rest)
        y_oob_bin = label_binarize(y_oob, classes=clases_unicas)
        auprc_oob = average_precision_score(y_oob_bin, y_prob_oob, average='weighted')
        
        # Almacenar métricas de esta iteración para el reporte final multiclase
        registro_metricas_oob.append({'Iteracion': i+1, 'F1_Macro_OOB': f1_macro_oob, 'AUPRC_OOB_Weighted': auprc_oob})
        
        # Marcar fin del ciclo y notificar progreso en consola
        fin_iter = time.time()
        print(f"   -> Iteración {i+1}/{n_repeticiones} completada en {round(fin_iter - inicio_iter, 1)} seg. | OOB F1: {f1_macro_oob:.4f}")

        # ==========================================================
        # AGREGADO DE SEGURIDAD (Destrucción de variables pesadas)
        # ==========================================================
        # Eliminar las matrices creadas en el ciclo actual para evitar desbordamientos de memoria
        del df_boot, df_oob, X_boot, y_boot, X_oob, y_oob
        # Forzar la limpieza de RAM profunda cada 10 iteraciones
        if (i + 1) % 10 == 0:
            gc.collect()

    # Evaluar tiempo total consumido por el proceso de Bootstrapping
    tiempo_fin_total = time.time()
    print(f"\n- Bootstrapping completado en {round((tiempo_fin_total - tiempo_inicio_total)/60, 2)} minutos.")

    # FASE 3: CONSOLIDACIÓN Y ALMACENAMIENTO DE RESULTADOS
    resultados_finales = []
    # Calcular promedios, varianzas y porcentajes consolidados para cada variable a lo largo del proceso
    for col in features:
        tasa_aparicion = (frecuencia_seleccion[col] / n_repeticiones) * 100
        imp_promedio = np.mean(memoria_importancias[col])
        imp_std = np.std(memoria_importancias[col])
        
        resultados_finales.append({
            'Variable': col,
            'Tasa_Seleccion_Porcentaje': tasa_aparicion,
            'Importancia_Promedio': imp_promedio,
            'Varianza_Importancia': imp_std
        })
        
    # Transformar la lista de resultados a DataFrame y ordenarlo por tasa de aparición (de mayor a menor)
    df_resultados = pd.DataFrame(resultados_finales)
    df_resultados = df_resultados.sort_values(by=['Tasa_Seleccion_Porcentaje', 'Importancia_Promedio'], ascending=[False, False])
    
    # Exportar el ranking de estabilidad de variables multiclase a CSV
    ruta_vars = os.path.join(dir_resultados, f"Estabilidad_Variables_XGB_{target_name}.csv")
    df_resultados.to_csv(ruta_vars, index=False)
    
    # Exportar el historial de rendimiento en Out-Of-Bag de cada iteración a CSV
    df_metricas = pd.DataFrame(registro_metricas_oob)
    ruta_metricas = os.path.join(dir_resultados, f"Metricas_OOB_XGB_{target_name}.csv")
    df_metricas.to_csv(ruta_metricas, index=False)
    
    # Imprimir resumen de guardado y métricas promedio finales por consola
    print("="*60)
    print(f"- Reporte de estabilidad guardado en: {ruta_vars}")
    print(f"- Métricas OOB guardadas en: {ruta_metricas}")
    print(f"Métrica OOB promedio final -> F1-Macro: {df_metricas['F1_Macro_OOB'].mean():.4f} | AUPRC Weighted: {df_metricas['AUPRC_OOB_Weighted'].mean():.4f}")
    print("="*60)

In [3]:
seleccion_caracteristicas_bootstrapping_rf_mortalidad()

INICIANDO BOOTSTRAPPING (100 ITERACIONES) PARA: MORTALIDAD (RANDOM FOREST)
Cargando datasets...
Balanceando la cohorte basal...
Iniciando bucle de 100 iteraciones (Nota: esto tomará tiempo)
   -> Iteración 1/100 completada en 413.1 seg. | OOB F1-Macro: 0.6888
   -> Iteración 2/100 completada en 395.0 seg. | OOB F1-Macro: 0.6915
   -> Iteración 3/100 completada en 394.9 seg. | OOB F1-Macro: 0.6901
   -> Iteración 4/100 completada en 400.4 seg. | OOB F1-Macro: 0.6871
   -> Iteración 5/100 completada en 396.6 seg. | OOB F1-Macro: 0.6877
   -> Iteración 6/100 completada en 396.3 seg. | OOB F1-Macro: 0.6883
   -> Iteración 7/100 completada en 394.7 seg. | OOB F1-Macro: 0.6896
   -> Iteración 8/100 completada en 396.0 seg. | OOB F1-Macro: 0.6916
   -> Iteración 9/100 completada en 393.5 seg. | OOB F1-Macro: 0.6906
   -> Iteración 10/100 completada en 401.7 seg. | OOB F1-Macro: 0.6887
   -> Iteración 11/100 completada en 390.9 seg. | OOB F1-Macro: 0.6898
   -> Iteración 12/100 completada en 3

In [4]:
seleccion_caracteristicas_bootstrapping_xgb_multiclase("SEVERIDAD")

INICIANDO BOOTSTRAPPING MULTICLASE PARA: SEVERIDAD (XGBOOST)
Cargando datasets...
Balanceando la cohorte basal...
Iniciando bucle de 100 iteraciones...
   -> Iteración 1/100 completada en 40.8 seg. | OOB F1: 0.7572
   -> Iteración 2/100 completada en 29.7 seg. | OOB F1: 0.7558
   -> Iteración 3/100 completada en 30.2 seg. | OOB F1: 0.7565
   -> Iteración 4/100 completada en 29.3 seg. | OOB F1: 0.7567
   -> Iteración 5/100 completada en 29.1 seg. | OOB F1: 0.7572
   -> Iteración 6/100 completada en 29.5 seg. | OOB F1: 0.7560
   -> Iteración 7/100 completada en 29.0 seg. | OOB F1: 0.7567
   -> Iteración 8/100 completada en 29.0 seg. | OOB F1: 0.7561
   -> Iteración 9/100 completada en 29.2 seg. | OOB F1: 0.7553
   -> Iteración 10/100 completada en 29.4 seg. | OOB F1: 0.7563
   -> Iteración 11/100 completada en 30.0 seg. | OOB F1: 0.7555
   -> Iteración 12/100 completada en 29.5 seg. | OOB F1: 0.7561
   -> Iteración 13/100 completada en 29.5 seg. | OOB F1: 0.7566
   -> Iteración 14/100 co

In [5]:
seleccion_caracteristicas_bootstrapping_xgb_multiclase("CONSUMO_RECURSOS")

INICIANDO BOOTSTRAPPING MULTICLASE PARA: CONSUMO_RECURSOS (XGBOOST)
Cargando datasets...
Balanceando la cohorte basal...
Iniciando bucle de 100 iteraciones...
   -> Iteración 1/100 completada en 28.1 seg. | OOB F1: 0.7485
   -> Iteración 2/100 completada en 23.3 seg. | OOB F1: 0.7478
   -> Iteración 3/100 completada en 23.7 seg. | OOB F1: 0.7488
   -> Iteración 4/100 completada en 22.9 seg. | OOB F1: 0.7471
   -> Iteración 5/100 completada en 23.0 seg. | OOB F1: 0.7483
   -> Iteración 6/100 completada en 22.3 seg. | OOB F1: 0.7472
   -> Iteración 7/100 completada en 23.9 seg. | OOB F1: 0.7489
   -> Iteración 8/100 completada en 24.5 seg. | OOB F1: 0.7478
   -> Iteración 9/100 completada en 23.6 seg. | OOB F1: 0.7481
   -> Iteración 10/100 completada en 23.6 seg. | OOB F1: 0.7477
   -> Iteración 11/100 completada en 24.0 seg. | OOB F1: 0.7471
   -> Iteración 12/100 completada en 23.3 seg. | OOB F1: 0.7480
   -> Iteración 13/100 completada en 23.0 seg. | OOB F1: 0.7479
   -> Iteración 14